**Objetivo**

Script responsável por criar uma tabela completa da alfabetização por municípios.

**Principais pontos**

**Fontes de dados**

- bronze.municipio

## 0. Configurando sessão spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# defina seu projeto para faturamento das queries
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_municipio = f"{par_source_project}.bronze.municipio"
par_source_silver_municipio = f"{par_source_project}.silver.municipio"

rede_map = {
    "0": "Total (Federal, Estadual, Municipal e Privada)",
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
    "5": "Pública (Estadual e Municipal)",
    "6": "Pública (Federal, Estadual e Municipal)",
}

## 3. Leitura dos dados da origem

In [5]:
df_scr_municipios = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_municipio)
    .load()
)

## 4. Transformações

In [6]:
df_municipios = (
    df_scr_municipios
    .select(
        'ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues',
        'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2',
        'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5',
        'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8'
    )
)
    

In [7]:
map_rede = F.create_map([F.lit(x) for kv in rede_map.items() for x in kv])

### 4.1. Chave de municípios

In [8]:
df_silver_municipios = (
    df_municipios
    # chave: código IBGE como string de 7 dígitos com zero à esquerda
    .withColumn("id_municipio", F.lpad(F.trim(F.col("id_municipio")), 7, "0"))
)

### 4.2. Padronização de dados

In [9]:
df_silver_municipios = (
    df_silver_municipios
    # tipos e padronização
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("serie", F.trim(F.col("serie")).cast("int"))
)

### 4.3. Descrição da coluna rede

In [10]:
df_silver_municipios = (
    df_silver_municipios
    # rede
    .withColumnRenamed("rede", "rede_id") # preserva o código original e adiciona o rótulo decodificado
    .withColumn("rede_id", F.trim(F.col("rede_id")))
    .withColumn("rede", map_rede[F.col("rede_id")])
)

### 4.4. Data de carregamento 

In [11]:
df_silver_municipios = (
    df_silver_municipios
    # auditoria da camada
    .withColumn("_silver_timestamp", F.current_timestamp())
)

### 4.5. Exclusão de duplicadas

In [13]:
# deduplicação pela chave natural
chave = ["ano", "id_municipio", "serie", "rede_id"]
df_silver_municipios = df_silver_municipios.dropDuplicates(chave)

## 5. Validação da qualidade

In [14]:
print("=== Relatório de Qualidade — df_silver_municipios.municipio ===")

# 1) Duplicidade na chave natural
qtd_bronze = df_scr_municipios.count()
chave_bronze = ["ano", "id_municipio", "serie", "rede"]
dups = df_scr_municipios.withColumn("id_municipio", F.lpad(F.trim("id_municipio"),7,"0")) \
             .groupBy(chave_bronze).count().filter("count > 1").count()
print(f"Duplicatas na chave (antes do dedup): {dups}")

# 2) Chaves inválidas (id_municipio deve ter 7 dígitos numéricos)
inval_id = df_silver_municipios.filter(~F.col("id_municipio").rlike("^[0-9]{7}$")).count()
print(f"id_municipio fora do padrão de 7 dígitos: {inval_id}")

# 3) Códigos de rede não mapeados (rede nula após o de-para = código desconhecido)
rede_nao_map = df_silver_municipios.filter(F.col("rede").isNull()).select("rede_id").distinct()
print("Códigos de rede NÃO mapeados (corrigir REDE_MAP):")
rede_nao_map.show()

# 4) Faixa de taxa_alfabetizacao (esperado 0–100)
fora_faixa = df_silver_municipios.filter((F.col("taxa_alfabetizacao") < 0) |
                           (F.col("taxa_alfabetizacao") > 100)).count()
print(f"taxa_alfabetizacao fora de [0,100]: {fora_faixa}")

# 5) Nulos nas colunas críticas
for c in ["ano", "id_municipio", "rede", "taxa_alfabetizacao", "media_portugues"]:
    n = df_silver_municipios.filter(F.col(c).isNull()).count()
    print(f"Nulos em coluna crítica '{c}': {n}")

print(f"Linhas Bronze: {qtd_bronze}  ->  Linhas Silver (pós-dedup): {df_silver_municipios.count()}")

=== Relatório de Qualidade — df_silver_municipios.municipio ===


Duplicatas na chave (antes do dedup): 0
id_municipio fora do padrão de 7 dígitos: 0
Códigos de rede NÃO mapeados (corrigir REDE_MAP):
+-------+
|rede_id|
+-------+
+-------+

taxa_alfabetizacao fora de [0,100]: 0
Nulos em coluna crítica 'ano': 0
Nulos em coluna crítica 'id_municipio': 0
Nulos em coluna crítica 'rede': 0
Nulos em coluna crítica 'taxa_alfabetizacao': 0
Nulos em coluna crítica 'media_portugues': 0
Linhas Bronze: 23995  ->  Linhas Silver (pós-dedup): 23995


## 6. Armazenamento no BQ

In [15]:
(
    df_silver_municipios.write.format("bigquery")
    .option("table", par_source_silver_municipio)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,rede_id")   
    .mode("overwrite")
    .save()
)

26/08/21 21:18:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                